In [12]:
from imageio.v3 import improps, imread

import csv
import os
import re

In [ ]:
def process(path):
    basename_pattern = re.compile("(.+)_C[0-9]+_traces_ID[0-9]+.csv")
    n_frames_pattern = re.compile("SizeX = ([0-9]+)")
    line_time_pattern = re.compile("Line time \\(s\\)\": ([0-9\\.]+)")

    # Iterating through files, only processing traces csvs
    obs_res_times = []
    for filename in os.listdir(path):
        if not filename.endswith(".csv") or "traces" not in filename:
            continue
        
        basename = basename_pattern.match(filename).group(1)
        
        # Getting number of frames
        im_props = improps(f"{path}{basename}.tif")
        line_time = 1/im_props.spacing[0]
        n_frames = im_props.shape[1]
        
        
        # Reading CSV
        with open(f"{path}{filename}", "r") as f:
            lines = f.readlines()
            start_frame = int(lines[1].split(",")[1])
            end_frame = int(lines[-2].split(",")[1])
            
            if end_frame > (n_frames-5):
                continue
            
            obs_res_time_frames = end_frame - start_frame
            if line_time is None:
                obs_res_time_s = None
            else:
                obs_res_time_s = obs_res_time_frames * line_time
            
            obs_res_times.append((filename,n_frames,obs_res_time_frames,obs_res_time_s))

    with open(f"{path}{path.split("/")[-2]}_observed_residence_times.csv", "w") as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "N frames","Observed_residence time (frames)", "Observed_residence time (s)"])
        for filename, n_frames, obs_res_time_frames, obs_res_time_s in obs_res_times:
            writer.writerow([filename, n_frames, obs_res_time_frames, obs_res_time_s])

In [14]:
path = '/Users/sc13967/Library/CloudStorage/OneDrive-UniversityofBristol/People/Gemma Fisher/2026-06-09 Observed residence time/2026_06_09_Observed residence time analysis for WT, EQ, trimer and elbow/WT/'

process(path)